In [ ]:
import os
import json
import pandas as pd
import numpy as np
import ast
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
from dotenv import load_dotenv
from groq import Groq

# ML imports
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, classification_report,
                              confusion_matrix)
from sklearn.preprocessing import LabelEncoder

# Plot settings
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['figure.dpi'] = 120
sns.set_theme(style="whitegrid")

load_dotenv()
client = Groq(api_key=os.getenv("GROQ_API_KEY"))

# Paths
DATA_PATH    = "/Users/ashishjain/Documents/assignment/transcript-intelligence/data/processed/all_calls.csv"
CHARTS_PATH  = "/Users/ashishjain/Documents/assignment/transcript-intelligence/outputs/charts/"
EXPORTS_PATH = "/Users/ashishjain/Documents/assignment/transcript-intelligence/data/exports/"

print("✅ Imports done")

In [ ]:
df = pd.read_csv(DATA_PATH)

# Fix list columns
df["topics"]       = df["topics"].apply(ast.literal_eval)
df["action_items"] = df["action_items"].apply(ast.literal_eval)
df["speakers"]     = df["speakers"].apply(ast.literal_eval)
df["key_moments"]  = df["key_moments"].apply(ast.literal_eval)
df["start_time"]   = pd.to_datetime(df["start_time"])

print(f"✅ Loaded   : {len(df)} calls")
print(f"📋 Columns  : {len(df.columns)}")
print(f"🏷️  Themes   : {df['theme'].nunique()}")

In [ ]:
print("=" * 60)
print("BONUS 1 — REPEAT ISSUES DETECTOR")
print("=" * 60)
print("Which problems keep coming back across calls?\n")

# Flatten all topics with call context
topic_records = []
for _, row in df.iterrows():
    for topic in row["topics"]:
        topic_records.append({
            "topic"     : topic,
            "call_type" : row["call_type"],
            "theme"     : row["theme"],
            "sentiment" : row["sentiment_score"],
            "date"      : row["start_time"]
        })

topics_df = pd.DataFrame(topic_records)

# Find repeat issues (appear 3+ times)
topic_freq = topics_df.groupby("topic").agg(
    frequency    = ("topic", "count"),
    avg_sentiment= ("sentiment", "mean"),
    call_types   = ("call_type", lambda x: list(x.unique()))
).reset_index().sort_values("frequency", ascending=False)

repeat_issues = topic_freq[topic_freq["frequency"] >= 3]

print(f"🔁 Topics appearing 3+ times : {len(repeat_issues)}")
print(f"\n🔝 Top 15 Repeat Issues:")
print(f"{'Topic':<35} {'Count':>6}  {'Avg Sentiment':>14}  Call Types")
print("─" * 75)
for _, row in repeat_issues.head(15).iterrows():
    types = ", ".join(row["call_types"])
    print(f"{row['topic']:<35} {row['frequency']:>6}x  "
          f"{row['avg_sentiment']:>12.2f}  {types}")

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

top15 = repeat_issues.head(15)
colors = ["#e74c3c" if s < 3.0 else
          "#f39c12" if s < 3.5 else
          "#2ecc71"
          for s in top15["avg_sentiment"]]

axes[0].barh(top15["topic"][::-1],
             top15["frequency"][::-1], color=colors[::-1])
axes[0].set_title("Top 15 Repeat Issues by Frequency",
                   fontweight="bold")
axes[0].set_xlabel("Times Mentioned")
for i, v in enumerate(top15["frequency"][::-1]):
    axes[0].text(v + 0.1, i, str(v), va="center", fontweight="bold")

# Repeat issues sentiment scatter
axes[1].scatter(repeat_issues["frequency"],
                repeat_issues["avg_sentiment"],
                c=["#e74c3c" if s < 3.0 else "#2ecc71"
                   for s in repeat_issues["avg_sentiment"]],
                s=80, alpha=0.7)
axes[1].axhline(3.0, color="gray", linestyle="--", linewidth=1)
axes[1].set_title("Repeat Issues: Frequency vs Sentiment",
                   fontweight="bold")
axes[1].set_xlabel("Frequency (times mentioned)")
axes[1].set_ylabel("Avg Sentiment Score")

# Label top 5 points
for _, row in repeat_issues.head(5).iterrows():
    axes[1].annotate(row["topic"][:20],
                     (row["frequency"], row["avg_sentiment"]),
                     textcoords="offset points",
                     xytext=(5, 5), fontsize=8)

plt.tight_layout()
plt.savefig(f"{CHARTS_PATH}repeat_issues.png", bbox_inches="tight")
plt.show()

# Save
repeat_issues.to_csv(f"{EXPORTS_PATH}repeat_issues.csv", index=False)
print("\n✅ Chart saved  → repeat_issues.png")
print("✅ Data saved   → data/exports/repeat_issues.csv")

In [ ]:
print("=" * 60)
print("BONUS 2 — ACTION ITEM BURDEN ANALYSIS")
print("=" * 60)
print("Who has the most follow-up load?\n")

# Extract action items with owner
action_records = []
for _, row in df.iterrows():
    for action in row["action_items"]:
        # Owner is usually before the colon
        owner = action.split(":")[0].strip() \
                if ":" in action else "Unknown"
        action_records.append({
            "owner"     : owner,
            "action"    : action,
            "call_type" : row["call_type"],
            "theme"     : row["theme"],
            "sentiment" : row["sentiment_score"],
            "title"     : row["title"]
        })

actions_df = pd.DataFrame(action_records)

print(f"📋 Total action items : {len(actions_df)}")
print(f"👥 Unique owners      : {actions_df['owner'].nunique()}")

# Owner burden
owner_burden = actions_df.groupby("owner").agg(
    total_actions  = ("action", "count"),
    avg_sentiment  = ("sentiment", "mean"),
    call_types     = ("call_type", lambda x: list(x.unique()))
).reset_index().sort_values("total_actions", ascending=False)

print(f"\n🏋️  Top 10 Action Item Owners (most burdened):")
print(f"{'Owner':<30} {'Actions':>8}  {'Avg Sentiment':>14}")
print("─" * 55)
for _, row in owner_burden.head(10).iterrows():
    print(f"{row['owner']:<30} {row['total_actions']:>8}  "
          f"{row['avg_sentiment']:>12.2f}")

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

top_owners = owner_burden.head(10)
colors = sns.color_palette("viridis", len(top_owners))
axes[0].barh(top_owners["owner"][::-1],
             top_owners["total_actions"][::-1],
             color=colors[::-1])
axes[0].set_title("Top 10 Most Burdened Action Item Owners",
                   fontweight="bold")
axes[0].set_xlabel("Total Action Items")
for i, v in enumerate(top_owners["total_actions"][::-1]):
    axes[0].text(v + 0.1, i, str(v), va="center", fontweight="bold")

# Action items by theme
theme_actions = df.groupby("theme")["num_action_items"] \
                  .mean().sort_values(ascending=False)
axes[1].bar(range(len(theme_actions)),
            theme_actions.values,
            color=sns.color_palette("magma", len(theme_actions)))
axes[1].set_xticks(range(len(theme_actions)))
axes[1].set_xticklabels(theme_actions.index,
                         rotation=35, ha="right", fontsize=9)
axes[1].set_title("Avg Action Items by Theme", fontweight="bold")
axes[1].set_ylabel("Avg Action Items per Call")
for i, v in enumerate(theme_actions.values):
    axes[1].text(i, v + 0.05, f"{v:.1f}",
                 ha="center", fontweight="bold", fontsize=9)

plt.tight_layout()
plt.savefig(f"{CHARTS_PATH}action_item_burden.png", bbox_inches="tight")
plt.show()

actions_df.to_csv(f"{EXPORTS_PATH}action_items.csv", index=False)
print("\n✅ Chart saved → action_item_burden.png")
print("✅ Data saved  → data/exports/action_items.csv")

In [ ]:
print("=" * 60)
print("BONUS 3 — SPEAKER TALK RATIO ANALYSIS")
print("=" * 60)
print("Who dominates conversations?\n")

# Load speakers data fresh from raw
import glob

speaker_records = []
DATA_RAW = "/Users/ashishjain/Documents/assignment/transcript-intelligence/data/raw/"

for folder in os.listdir(DATA_RAW):
    folder_path = os.path.join(DATA_RAW, folder)
    if not os.path.isdir(folder_path):
        continue
    try:
        with open(f"{folder_path}/speakers.json") as f:
            speakers = json.load(f)
        with open(f"{folder_path}/meeting-info.json") as f:
            meeting = json.load(f)

        # Match to df for call_type
        meeting_id = meeting.get("meetingId")
        call_row   = df[df["meeting_id"] == meeting_id]
        call_type  = call_row["call_type"].values[0] \
                     if len(call_row) > 0 else "Unknown"

        for s in speakers:
            duration = s.get("endTimeTs", 0) - s.get("timestamp", 0)
            speaker_records.append({
                "meeting_id"    : meeting_id,
                "speaker"       : s.get("speakerName"),
                "talk_duration" : duration,
                "call_type"     : call_type
            })
    except Exception as e:
        continue

speakers_df = pd.DataFrame(speaker_records)

# Talk ratio per call
talk_ratio = speakers_df.groupby(
    ["meeting_id", "speaker", "call_type"]
)["talk_duration"].sum().reset_index()

talk_ratio["total_call_duration"] = talk_ratio.groupby(
    "meeting_id")["talk_duration"].transform("sum")
talk_ratio["talk_pct"] = (
    talk_ratio["talk_duration"] /
    talk_ratio["total_call_duration"] * 100
).round(1)

print(f"📊 Avg talk ratio by call type:")
avg_ratio = talk_ratio.groupby("call_type")["talk_pct"].mean()
for ctype, pct in avg_ratio.items():
    print(f"   {ctype:<25} → {pct:.1f}% avg per speaker")

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Talk ratio distribution
axes[0].hist(talk_ratio["talk_pct"], bins=20,
             color="#534AB7", edgecolor="white")
axes[0].axvline(50, color="red", linestyle="--",
                linewidth=1.5, label="50% (equal)")
axes[0].set_title("Talk Ratio Distribution", fontweight="bold")
axes[0].set_xlabel("% of Call Talking")
axes[0].set_ylabel("Count")
axes[0].legend()

# By call type boxplot
talk_ratio.boxplot(column="talk_pct", by="call_type", ax=axes[1])
axes[1].set_title("Talk Ratio by Call Type", fontweight="bold")
axes[1].set_xlabel("Call Type")
axes[1].set_ylabel("% Talk Time")
plt.sca(axes[1])
plt.xticks(rotation=15)

plt.tight_layout()
plt.savefig(f"{CHARTS_PATH}speaker_talk_ratio.png",
            bbox_inches="tight")
plt.show()
print("\n✅ Chart saved → speaker_talk_ratio.png")

In [ ]:
print("=" * 60)
print("BONUS 4 — CHURN RISK PREDICTOR (ML MODEL)")
print("=" * 60)
print("Random Forest Classifier — 80/20 split\n")

# ── Feature Engineering ──────────────────────────
le_type  = LabelEncoder()
le_theme = LabelEncoder()

df["call_type_enc"] = le_type.fit_transform(df["call_type"])
df["theme_enc"]     = le_theme.fit_transform(df["theme"])

# Define churn risk label
# High risk = sentiment < 3.0 OR very-negative/negative label
df["churn_risk"] = (
    (df["sentiment_score"] < 3.0) |
    (df["sentiment"].isin(["very-negative", "negative"]))
).astype(int)

print(f"📊 Churn Risk Distribution:")
print(f"   At Risk (1) : {df['churn_risk'].sum()} accounts")
print(f"   Safe    (0) : {(df['churn_risk']==0).sum()} accounts")

# ── Features & Target ────────────────────────────
features = [
    "sentiment_score",
    "num_action_items",
    "positive_sents",
    "negative_sents",
    "neutral_sents",
    "duration_mins",
    "num_sentences",
    "num_speakers",
    "call_type_enc",
    "theme_enc"
]

X = df[features]
y = df["churn_risk"]

# ── Train/Test Split (80/20) ─────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"\n📊 Train/Test Split:")
print(f"   Training set : {len(X_train)} calls (80%)")
print(f"   Test set     : {len(X_test)} calls  (20%)")

# ── Train Random Forest ───────────────────────────
rf_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    max_depth=5
)
rf_model.fit(X_train, y_train)
rf_preds = rf_model.predict(X_test)
rf_acc   = accuracy_score(y_test, rf_preds)

# ── Train Logistic Regression (comparison) ───────
lr_model = LogisticRegression(random_state=42, max_iter=1000)
lr_model.fit(X_train, y_train)
lr_preds = lr_model.predict(X_test)
lr_acc   = accuracy_score(y_test, lr_preds)

print(f"\n🎯 Model Results:")
print(f"   Random Forest Accuracy     : {rf_acc*100:.1f}%")
print(f"   Logistic Regression Accuracy: {lr_acc*100:.1f}%")

print(f"\n📋 Random Forest Classification Report:")
print(classification_report(y_test, rf_preds,
      target_names=["Safe", "At Risk"]))

# ── Visualize ─────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Feature importance
feat_imp = pd.Series(rf_model.feature_importances_,
                     index=features).sort_values(ascending=True)
feat_imp.plot(kind="barh", ax=axes[0],
              color=sns.color_palette("viridis", len(features)))
axes[0].set_title("Feature Importance (Random Forest)",
                   fontweight="bold")
axes[0].set_xlabel("Importance Score")

# Confusion matrix
cm = confusion_matrix(y_test, rf_preds)
sns.heatmap(cm, annot=True, fmt="d", cmap="Reds",
            ax=axes[1],
            xticklabels=["Safe", "At Risk"],
            yticklabels=["Safe", "At Risk"])
axes[1].set_title("Confusion Matrix", fontweight="bold")
axes[1].set_xlabel("Predicted")
axes[1].set_ylabel("Actual")

plt.tight_layout()
plt.savefig(f"{CHARTS_PATH}churn_risk_model.png",
            bbox_inches="tight")
plt.show()

# Add predictions back to df
df["churn_probability"] = rf_model.predict_proba(X)[:, 1]
df["predicted_churn"]   = rf_model.predict(X)
print(f"\n✅ Chart saved → churn_risk_model.png")

In [ ]:
print("=" * 60)
print("BONUS INSIGHTS — FINAL SUMMARY")
print("=" * 60)

top_repeat  = repeat_issues.iloc[0]
top_owner   = owner_burden.iloc[0]
churn_count = df["predicted_churn"].sum()

print(f"""
📌 BONUS 1 — Repeat Issues
   Most repeated issue  : {top_repeat['topic']}
   Mentioned            : {top_repeat['frequency']}x across calls
   Avg sentiment        : {top_repeat['avg_sentiment']:.2f}/5.0
   → Same problems recurring = systemic, not one-off

📌 BONUS 2 — Action Item Burden
   Most burdened person : {top_owner['owner']}
   Total action items   : {top_owner['total_actions']}
   → Uneven workload distribution detected

📌 BONUS 3 — Speaker Talk Ratio
   → Imbalanced talk ratios in support calls
   → Agents may not be listening enough
   → Active listening training recommended

📌 BONUS 4 — Churn Risk ML Model
   Algorithm            : Random Forest Classifier
   Train/Test Split     : 80% / 20%
   Accuracy             : {rf_acc*100:.1f}%
   Predicted at-risk    : {churn_count} accounts
   Top feature          : {feat_imp.index[-1]} (most important)
   → Model identifies at-risk accounts proactively
""")

# Save final DataFrame
df.to_csv(DATA_PATH, index=False)
df[["title", "call_type", "theme", "sentiment",
    "sentiment_score", "churn_risk",
    "churn_probability", "risk_level"]].to_csv(
    f"{EXPORTS_PATH}final_analysis.csv", index=False)

print("✅ Final data → data/processed/all_calls.csv")
print("✅ Analysis   → data/exports/final_analysis.csv")
print("\n🚀 Ready for 06_visualizations.ipynb!")